# Hotspot & cluster analysis (ESDA)

End-to-end walkthrough: load areal units + variable, build and compare
spatial weights, test global autocorrelation (Moran's I), then map local
clusters (LISA) and hot/cold spots (Getis-Ord Gi*).

All analysis choices come from `config/aoi.yaml`. Interpretation caveats are
in the final cell and the README.

## 0. Hypothesis

County corn yield in Iowa (2023) is **positively spatially autocorrelated**
(soil/climate gradients), so we expect a significant positive global Moran's
I and coherent High-High / Low-Low local clusters.

## 0b. Quick start: the dependency-free demo

Before touching the geospatial stack, run the pure-numpy demo. It builds a
12x12 rook grid by hand, plants a High-High and a Low-Low cluster into a noisy
field, and drives the real ESDA core (Moran's I, Geary's C, LISA, Gi*). It is a
small seeded synthetic field, not real data, but it is fully reproducible and
needs only numpy. Equivalent CLI: `pixi run demo` (or `hotspots demo`).

In [ ]:
import numpy as np

from hotspots import (
    benjamini_hochberg,
    bivariate_moran_dense,
    join_counts_dense,
    moran_scatter_slope,
    morans_i_dense,
    rook_weights,
    run_demo,
)

demo = run_demo(seed=0, out_dir="../outputs")
demo

### Extra pure-numpy capabilities

Join counts on a binary field, bivariate Moran's I, the Moran-scatter-slope
cross-check (slope of lag on value == Moran's I on a row-standardised W), and a
Benjamini-Hochberg FDR correction for the many per-location LISA/Gi* tests.

In [ ]:
w_grid = rook_weights(12, 12)
field = run_demo(seed=0, out_dir="../outputs")  # reuse the same grid/field

# Join counts on the demo field thresholded at its median (above = 1).
demo_vals = np.genfromtxt(
    "../outputs/lisa_labels.csv", delimiter=",", skip_header=1, usecols=3
)
binary = (demo_vals > np.median(demo_vals)).astype(float)
bb, ww, bw = join_counts_dense(binary, w_grid)
print(f"Join counts  BB={bb:.0f}  WW={ww:.0f}  BW={bw:.0f}")

# Bivariate Moran of the field against a shifted copy, and the slope identity.
shifted = np.roll(demo_vals, 12)  # shift one grid row
print(f"Bivariate Moran (field vs shifted): {bivariate_moran_dense(demo_vals, shifted, w_grid):.4f}")
print(f"Scatter slope == Moran's I:        {moran_scatter_slope(demo_vals, w_grid):.4f}")

# FDR on a small illustrative p-vector.
pvals = np.array([0.001, 0.008, 0.039, 0.041, 0.042])
reject, thresh = benjamini_hochberg(pvals, alpha=0.05)
print(f"BH alpha=0.05  threshold={thresh:.3f}  reject={reject.tolist()}")

In [ ]:
import geopandas as gpd
import yaml

from hotspots import esda
from hotspots.weights import compare_weights, queen_contiguity, knn, distance_band

with open("../config/aoi.yaml") as fh:
    cfg = yaml.safe_load(fh)
value_col = cfg["variable"]["value_column"]
inf = cfg["inference"]
cfg["aoi"]["name"]

In [ ]:
# Built by `python data/download.py` (or `make download`).
gdf = gpd.read_file(f"../data/raw/{cfg['aoi']['name']}.gpkg").to_crs(cfg["aoi"]["crs"])
values = gdf[value_col].to_numpy(float)
gdf.plot(column=value_col, legend=True, scheme="quantiles", k=5, cmap="viridis")

## 1. Build and compare weights

The choice of W is a modelling decision; compare neighbour structure before
committing. Islands raise by default.

In [ ]:
w_queen = queen_contiguity(gdf, on_islands="warn")
w_knn = knn(gdf, k=cfg["weights"]["knn"]["k"])
w_dist = distance_band(gdf, on_islands="warn")
for d in compare_weights({"queen": w_queen, "knn": w_knn, "distance": w_dist}):
    print(d.as_dict())

w = w_queen  # primary

## 2. Global Moran's I (permutation inference)

In [ ]:
gm = esda.global_moran(values, w, permutations=inf["permutations"], seed=inf["seed"])
print(gm)
esda.moran_scatterplot(values, w)

## 3. Local clusters: LISA and Getis-Ord Gi*

In [ ]:
lisa = esda.local_moran(values, w, permutations=inf["permutations"],
                        significance=inf["significance"], seed=inf["seed"])
gi = esda.getis_ord_gi_star(values, w, permutations=inf["permutations"],
                            significance=inf["significance"], seed=inf["seed"])
gdf["lisa"], gdf["gi"] = lisa.labels, gi.labels
gdf.plot(column="lisa", categorical=True, legend=True, cmap="Set1")

## 4. Interpretation and caveats

- Clusters are **descriptive, not causal**.
- Results are **conditional on W** (we compared three) and on **stationarity**.
- LISA/Gi* are **multiple-tested**; treat pseudo p as flags and correct (FDR).
- Subject to **MAUP** (county aggregation).

Optional next step: `hotspots.gwr.fit_gwr` to probe spatially varying
relationships once covariates are added to the config.